# First-order properties

First-order properties quantify expectation values of one-electron operators in individual electronic states. For the electronic ground state, quantities such as the permanent electric dipole moment are evaluated directly from the converged Kohn–Sham density, requiring no response treatment. In contrast, expectation values for electronically excited states—most notably excited-state dipole moments—are obtained from the double residue of the quadratic response function, which provides the transition-specific way to construct expectation values in the excited-state manifold. Together, these formulations provide a consistent route to state-specific first-order properties across both ground and excited electronic states. See {cite}`Norman2018` for further details.

## Electric dipole moment

The electric dipole moment consists of separate nuclear and electronic contributions, arising from the distribution of nuclear charges and the electronic density, respectively. VeloxChem evaluates the total dipole moment as the sum of these two components. For charge-neutral systems, the total dipole moment is independent of the choice of gauge origin. In contrast, for ionic systems the dipole moment depends on the origin of the coordinate system. In such cases, VeloxChem reports dipole moments computed with respect to the center of nuclear charge, providing a well-defined and physically meaningful reference point.

The standard unit for electric dipole moments is Debye (D) after [Peter Debye](https://en.wikipedia.org/wiki/Peter_Debye). The conversion factor between atomic units and Debye is

$$
1 \; e a_0 = 2.541746 \; \mathrm{D}
$$

### Ground state

**Python script**

In [45]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("para-nitroaniline")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "b3lyp"
scf_results = scf_drv.compute(molecule, basis)

prop_drv = vlx.FirstOrderPropertyDriver()

prop_drv.property = "electric dipole moment"

prop_results = prop_drv.compute(molecule, basis, scf_results)

Reading para-nitroaniline from PubChem...

Reference: S. Kim, J. Chen, T. Cheng, A. Gindulyte, J. He, S. He, Q. Li, B. A. Shoemaker, P. A. Thiessen, B. Yu, L. Zaslavsky, J. Zhang, E. E. Bolton, Nucleic Acids Res., 2025, 53, D1516-D1525.

Please double-check the compound since names may refer to more than one record.

                                                                                                                          
                                            Self Consistent Field Driver Setup                                            
                                                                                                                          
                   Wave Function Model             : Spin-Restricted Kohn-Sham                                            
                   Initial Guess Model             : Superposition of Atomic Densities                                    
                   Convergence Accelerator         : Two Level Dir

In [51]:
import numpy as np

results = prop_results["electric dipole moment"]

dipmom = np.linalg.norm(results["total"])
au2debye = 2.541746

print(f"Electric dipole moment: {dipmom * au2debye: .4f} Debye")

print(f"Components (a.u.):\n{"x":>6s} {"y":>8s} {"z":>8s}")
print(
    f"{results["total"][0]:>8.4f} {results["total"][1]:>8.4f} {results["total"][2]:>8.4f}"
)

Electric dipole moment:  8.2621 Debye
Components (a.u.):
     x        y        z
 -1.0103   2.7091  -1.4852


In [50]:
molecule.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### Excited state

**Python script**

In [ ]:
import veloxchem as vlx

molecule = vlx.Molecule.read_name("para-nitroaniline")
basis = vlx.MolecularBasis.read(molecule, "def2-svp")

scf_drv = vlx.ScfRestrictedDriver()
scf_drv.xcfun = "cam-b3lyp"
scf_results = scf_drv.compute(molecule, basis)

In [4]:
from veloxchem.doubleresbeta import DoubleResBetaDriver

In [5]:
excmom_drv = DoubleResBetaDriver()

excmom_drv.initial_state = 3
excmom_drv.final_state = 3

excmom_results = excmom_drv.compute(molecule, basis, scf_results)

                                                                                                                          
                                             Quadratic Response Driver Setup                                              
                                                                                                                          
                                    ERI Screening Threshold         : 1.0e-12                                             
                                    Convergance Threshold           : 1.0e-04                                             
                                    Max. Number of Iterations       : 150                                                 
                                    Max. Number of Iterations       : 150                                                 
                                                                                                                          
                

In [40]:
mu_g = np.array(excmom_results['ground_state_dipole_moments'])

mu = []
for key, value in excmom_results['excited_state_dipole_moments'].items():
    mu.append(value)

mu = np.array(mu)

mu_e = -(mu - 2 * mu_g)
print(f"\nElectric dipole moment: {np.linalg.norm(mu_e) * au2debye: .4f} Debye")


Dipole moment:  12.7337 Debye


**Text file**

:::{code}
jobs
task: response
@end

@method settings
xcfun: cam-b3lyp
basis: def2-svpd
@end

@response
property: transition dipole moment
initial_state: 1
final_state: 1
@end

@molecule
charge: 0
multiplicity: 1
xyz:
...
@end
:::
